In [1]:
import os
import glob
import gc

from pathlib import Path

import numpy as np
import pandas as pd
import h5py

from scipy.sparse import csr_matrix
from pathlib import Path

import scipy

from scipy.stats import median_abs_deviation, percentileofscore

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns

import anndata as ad
import scanpy.external as sce
import scanpy as sc

import pegasus as pg
from pegasusio import UnimodalData, MultimodalData
import scrublet as scr

from tqdm import tqdm

%matplotlib inline

sc.settings.n_jobs = 25

In [2]:
FERNANDO_SRR_FOLDERS = glob.glob('/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/star_outputs/*_Solo.out/')

SHIN_SRR_FOLDERS = glob.glob('/mnt/sdb/scz_meta_analysis_processed/shin_nowakowski/star_outputs/*_Solo.out/')

NOTARAS_SRR_FOLDERS = glob.glob('/mnt/sdb/scz_meta_analysis_processed/notaras_colak/star_outputs/*_Solo.out/')

WALSH_SRR_FOLDERS = glob.glob('/mnt/sdb/scz_meta_analysis_processed/walsh_studer/star_outputs/*_Solo.out/')

RAO_SRR_FOLDERS = glob.glob('/mnt/sdb/scz_meta_analysis_processed/rao_gogos/star_outputs/*_Solo.out/')

SEBASTIAN_SRR_FOLDERS = glob.glob('/mnt/sdb/scz_meta_analysis_processed/sebastian_pak/star_outputs/*_Solo.out/')

SAWADA_SRR_FOLDERS = glob.glob('/mnt/sdb/scz_meta_analysis_processed/sawada_kato/star_outputs/*_Solo.out/')

In [ ]:
ALL_SRR_FOLDERS = (FERNANDO_SRR_FOLDERS + SHIN_SRR_FOLDERS + NOTARAS_SRR_FOLDERS + WALSH_SRR_FOLDERS + 
                   RAO_SRR_FOLDERS + SEBASTIAN_SRR_FOLDERS + SAWADA_SRR_FOLDERS)

In [10]:
# ## Functions Iterate Through Folders And Build H5 ##

def assemble_library_annata(srr_folder):
    
    ''' takes the path to an SRR_Solo.out/ and generates the anndata object '''
    
    raw_path = Path(srr_folder) / "Gene/raw"
    adata = sc.read_mtx(raw_path / "matrix.mtx").T
    adata.var = pd.read_csv(raw_path / "features.tsv", delimiter="\t", header=None, names=['gene_symbols', 'assay'])
    adata.obs = pd.read_csv(raw_path / "barcodes.tsv", header=None, index_col=0, names=['barcodes'])
    adata.obs_names = [srr_folder.split('/')[-2].split('_')[0] + '_' + obs_name for obs_name in adata.obs_names] 
    adata.obs['library'] = srr_folder.split('/')[-2].split('_')[0]
    
    return adata

In [38]:
# ## Build H5 Off Adata For Cellbender Input ##

# def write_10X_h5(adata, file):
#     """Writes adata to a 10X-formatted h5 file.
    
#     Note that this function is not fully tested and may not work for all cases.
#     It will not write the following keys to the h5 file compared to 10X:
#     '_all_tag_keys', 'pattern', 'read', 'sequence'

#     Args:
#         adata (AnnData object): AnnData object to be written.
#         file (str): File name to be written to. If no extension is given, '.h5' is appended.

#     Raises:
#         FileExistsError: If file already exists.

#     Returns:
#         None
#     """
    
#     if '.h5' not in file: file = f'{file}.h5'
#     if Path(file).exists():
#         raise FileExistsError(f"There already is a file `{file}`.")
#     def int_max(x):
#         return int(max(np.floor(len(str(int(max(x)))) / 4), 1) * 4)
#     def str_max(x):
#         return max([len(i) for i in x])

#     w = h5py.File(file, 'w')
#     grp = w.create_group("matrix")
#     grp.create_dataset("barcodes", data=np.array(adata.obs_names, dtype=f'|S{str_max(adata.obs_names)}'))
#     grp.create_dataset("data", data=np.array(adata.X.data, dtype=f'<i{int_max(adata.X.data)}'))
#     ftrs = grp.create_group("features")
#     # this group will lack the following keys:
#     # '_all_tag_keys', 'feature_type', 'genome', 'id', 'name', 'pattern', 'read', 'sequence'
#     #ftrs.create_dataset("feature_type", data=np.array(adata.var.feature_types, dtype=f'|S{str_max(adata.var.feature_types)}'))
#     #ftrs.create_dataset("genome", data=np.array(adata.var.genome, dtype=f'|S{str_max(adata.var.genome)}'))
#     #ftrs.create_dataset("id", data=np.array(adata.var.gene_ids, dtype=f'|S{str_max(adata.var.gene_ids)}'))
#     ftrs.create_dataset("name", data=np.array(adata.var.index, dtype=f'|S{str_max(adata.var.index)}'))
#     grp.create_dataset("indices", data=np.array(adata.X.indices, dtype=f'<i{int_max(adata.X.indices)}'))
#     grp.create_dataset("indptr", data=np.array(adata.X.indptr, dtype=f'<i{int_max(adata.X.indptr)}'))
#     grp.create_dataset("shape", data=np.array(list(adata.X.shape)[::-1], dtype=f'<i{int_max(adata.X.shape)}'))

In [39]:
for srr_folder in tqdm(ALL_SRR_FOLDERS):
    SRR_NAME = srr_folder.split('/')[-2].split('_')[0]
    adata = None
    adata = assemble_library_annata(srr_folder)
    write_10X_h5(adata, srr_folder + 'Gene/raw/{0}.h5'.format(SRR_NAME))
    del adata
    gc.collect()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 241/241 [26:32<00:00,  6.61s/it]


In [ ]:
## Write a Script that TAkes CellBender and Deploys it on each set of SRR Folders ##

In [61]:
import h5py
import numpy as np
from pathlib import Path
import scipy.sparse
import os

def write_10X_h5(adata, file):
    """Writes adata to a 10X-formatted h5 file."""
    
    if '.h5' not in file:
        file = f'{file}.h5'
    
    # If the file exists, remove it
    if Path(file).exists():
        os.remove(file)  # Delete the existing file
        
    # Ensure the sparse matrix is in CSR format
    if isinstance(adata.X, scipy.sparse.spmatrix):
        adata.X = adata.X.tocsr()

    def int_max(x):
        return int(max(np.floor(len(str(int(max(x)))) / 4), 1) * 4)
    
    def str_max(x):
        return max([len(i) for i in x])

    # No need to transpose, as it's already cells x genes
    # Proceed to write the file

    with h5py.File(file, 'w') as w:
        grp = w.create_group("matrix")
        grp.create_dataset("barcodes", data=np.array(adata.obs_names, dtype=f'|S{str_max(adata.obs_names)}'))
        ftrs = grp.create_group("features")
        ftrs.create_dataset("name", data=np.array(adata.var.index, dtype=f'|S{str_max(adata.var.index)}'))
        
        # Writing the sparse matrix
        grp.create_dataset("data", data=np.array(adata.X.data, dtype='<f4'))  # Use float32 ('<f4') for dense data
        grp.create_dataset("indices", data=np.array(adata.X.indices, dtype='<i4'))  # Use int32 ('<i4') for indices
        grp.create_dataset("indptr", data=np.array(adata.X.indptr, dtype='<i4'))  # Use int32 ('<i4') for indptr
        grp.create_dataset("shape", data=np.array(adata.X.shape, dtype='<i4'))  # Use int32 ('<i4') for shape


In [63]:
adata.X.shape[0] == len(adata.var)

False

In [46]:
ALL_SRR_FOLDERS[0]

'/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/star_outputs/SRR32261339_Solo.out/'

In [44]:
adata = assemble_library_annata(ALL_SRR_FOLDERS[0])

In [62]:
write_10X_h5(adata, '/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/star_outputs/SRR32261339_Solo.out/Gene/raw/SRR32261339.h5')

In [49]:
adata

AnnData object with n_obs × n_vars = 3686400 × 33538
    obs: 'library'
    var: 'gene_symbols', 'assay'

In [8]:
## Load Khan-Pasca ##

glob.glob('/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/*')

['/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306931_Control_1_matrix.mtx.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306931_Control_1_features.tsv.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306934_Patient_2_matrix.mtx.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306931_Control_1_barcodes.tsv.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306933_Patient_1_matrix.mtx.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306932_Control_2_matrix.mtx.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306933_Patient_1_features.tsv.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306934_Patient_2_barcodes.tsv.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/GSM4306933_Patient_1_barcodes.tsv.gz',
 '/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_fil

In [28]:
## Khan Pasca H5 Construction ##

## Khan 10x Files Import ##

KHAN_GEO_PATH = Path('/mnt/sda/scz_meta_analysis/khan_pasca_nature_med_2021/GEO_files/')

#CTRL1
CTR1 = sc.read_mtx(KHAN_GEO_PATH / "GSM4306931_Control_1_matrix.mtx.gz").T
CTR1.var = pd.read_csv(KHAN_GEO_PATH / "GSM4306931_Control_1_features.tsv.gz", delimiter="\t", header=None, names=['gene_symbols', 'assay'])
CTR1.obs = pd.read_csv(KHAN_GEO_PATH / "GSM4306931_Control_1_barcodes.tsv.gz", header=None, index_col=0, names=['barcodes'])
CTR1.obs['library'] = 'GSM4306931_Control_1'
CTR1.obs_names = ['GSM4306931_Control_1_' + obs_name for obs_name in CTR1.obs_names] 


#CTRL2
CTR2 = sc.read_mtx(KHAN_GEO_PATH / "GSM4306932_Control_2_matrix.mtx.gz").T
CTR2.var = pd.read_csv(KHAN_GEO_PATH / "GSM4306932_Control_2_features.tsv.gz", delimiter="\t", header=None, names=['gene_symbols', 'assay'])
CTR2.obs = pd.read_csv(KHAN_GEO_PATH / "GSM4306932_Control_2_barcodes.tsv.gz", header=None, index_col=0, names=['barcodes'])
CTR2.obs['library'] = 'GSM4306932_Control_2'
CTR2.obs_names = ['GSM4306932_Control_2_' + '_' + obs_name for obs_name in CTRL2.obs_names] 

#PT1
PT1 = sc.read_mtx(KHAN_GEO_PATH / "GSM4306933_Patient_1_matrix.mtx.gz").T
PT1.var = pd.read_csv(KHAN_GEO_PATH / "GSM4306933_Patient_1_features.tsv.gz", delimiter="\t", header=None, names=['gene_symbols', 'assay'])
PT1.obs = pd.read_csv(KHAN_GEO_PATH / "GSM4306933_Patient_1_barcodes.tsv.gz", header=None, index_col=0, names=['barcodes'])
PT1.obs['library'] = 'GSM4306933_Patient_1'
PT1.obs_names = ['GSM4306933_Patient_1_' + '_' + obs_name for obs_name in PT1.obs_names] 

#PT2
PT2 = sc.read_mtx(KHAN_GEO_PATH / "GSM4306934_Patient_2_matrix.mtx.gz").T
PT2.var = pd.read_csv(KHAN_GEO_PATH / "GSM4306934_Patient_2_features.tsv.gz", delimiter="\t", header=None, names=['gene_symbols', 'assay'])
PT2.obs = pd.read_csv(KHAN_GEO_PATH / "GSM4306934_Patient_2_barcodes.tsv.gz", header=None, index_col=0, names=['barcodes'])
PT2.obs['library'] = 'GSM4306934_Patient_2'
PT2.obs_names = ['GSM4306934_Patient_2_' + '_' + obs_name for obs_name in PT2.obs_names] 

## Concatenate Files ##

khan_adata = sc.concat([CTR1, CTR2, PT1, PT2], join='outer', label="library")
khan_adata.var['feature_types'] = 'Gene Expression'

In [28]:
write_10X_h5(fernando_adata, '/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/cellbender/fernando.h5')

In [34]:
write_10X_h5(sawada_adata, '/mnt/sdb/scz_meta_analysis_processed/sawada_kato/cellbender/sawada.h5')

In [ ]:
write_10X_h5(rao_adata, '/mnt/sdb/scz_meta_analysis_processed/rao_gogos/cellbender/rao.h5')

In [ ]:
write_10X_h5(notaras_adata, '/mnt/sdb/scz_meta_analysis_processed/notaras_colak/cellbender/notaras.h5')

In [ ]:
write_10X_h5(sebastian_adata, '/mnt/sdb/scz_meta_analysis_processed/sebastian_pak/cellbender/sebastian.h5')

In [39]:
glob.glob(FERNANDO_SRR_FOLDERS[0] + 'Gene/raw/*')

['/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/star_outputs/SRR32261339_Solo.out/Gene/raw/features.tsv',
 '/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/star_outputs/SRR32261339_Solo.out/Gene/raw/matrix.mtx',
 '/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/star_outputs/SRR32261339_Solo.out/Gene/raw/barcodes.tsv']

In [57]:
TEST_PATH = Path('/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/star_outputs/SRR32261339_Solo.out/Gene/raw/')

TEST = sc.read_mtx(TEST_PATH / "matrix.mtx").T
TEST.var = pd.read_csv(TEST_PATH / "features.tsv", delimiter="\t", header=None, names=['gene_symbols', 'assay'])
TEST.obs = pd.read_csv(TEST_PATH / "barcodes.tsv", header=None, index_col=0, names=['barcodes'])
TEST.obs['library'] = 'SRR32261339'
TEST.obs_names = ['SRR32261339' + '_' + obs_name for obs_name in TEST.obs_names] 
TEST.var['feature_types'] = 'Gene Expression'

In [54]:
TEST.write(TEST_PATH / 'cellbender_test.h5')

In [69]:
import numpy as np
import h5py
import scanpy as sc
from pathlib import Path
from scipy.sparse import csc_matrix, csr_matrix

def write_10X_h5(adata, file):
    """Writes adata to a 10X-formatted h5 file for CellBender input."""
    if '.h5' not in file: file = f'{file}.h5'
    if Path(file).exists():
        raise FileExistsError(f"There already is a file `{file}`.")
    
    def int_max(x):
        return int(max(np.floor(len(str(int(max(x)))) / 4), 1) * 4)

    def str_max(x):
        return max([len(i) for i in x])

    # Ensure adata.X is in CSR format (convert if necessary)
    if isinstance(adata.X, csc_matrix):
        adata.X = adata.X.tocsr()  # Convert from CSC to CSR format

    w = h5py.File(file, 'w')
    grp = w.create_group("matrix")
    
    # Write barcodes and features
    grp.create_dataset("barcodes", data=np.array(adata.obs_names, dtype=f'|S{str_max(adata.obs_names)}'))
    ftrs = grp.create_group("features")
    
    # Write feature types, gene names, and id
    ftrs.create_dataset("feature_type", data=np.array(adata.var.feature_types, dtype=f'|S{str_max(adata.var.feature_types)}'))
    ftrs.create_dataset("name", data=np.array(adata.var.index, dtype=f'|S{str_max(adata.var.index)}'))
    ftrs.create_dataset("id", data=np.array(adata.var.index, dtype=f'|S{str_max(adata.var.index)}'))  # Use gene symbols as 'id'
    
    # Extract sparse matrix components
    data = adata.X.data
    indices = adata.X.indices
    indptr = adata.X.indptr
    shape = adata.X.shape

    # Write sparse matrix components
    grp.create_dataset("data", data=np.array(data, dtype='float32'))  # Ensure float32 for sparse data
    grp.create_dataset("indices", data=np.array(indices, dtype='int32'))  # Indices as integers
    grp.create_dataset("indptr", data=np.array(indptr, dtype='int32'))  # Indptr as integers
    grp.create_dataset("shape", data=np.array(list(shape)[::-1], dtype='int32'))  # Reverse for row-major order

    w.close()

# Test the function
TEST_PATH = Path('/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/star_outputs/SRR32261339_Solo.out/Gene/raw/')

TEST = sc.read_mtx(TEST_PATH / "matrix.mtx").T
TEST.var = pd.read_csv(TEST_PATH / "features.tsv", delimiter="\t", header=None, names=['gene_symbols', 'assay'])
TEST.obs = pd.read_csv(TEST_PATH / "barcodes.tsv", header=None, index_col=0, names=['barcodes'])
TEST.obs['library'] = 'SRR32261339'
TEST.obs_names = ['SRR32261339' + '_' + obs_name for obs_name in TEST.obs_names] 
TEST.var['feature_types'] = 'Gene Expression'

write_10X_h5(TEST, '/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/cellbender/cellbender_test.h5')


In [58]:
# write_10X_h5(TEST, '/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/cellbender/cellbender_test.h5')

In [70]:
adata = sc.read_10x_h5('/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/cellbender/cellbender_test.h5')

In [77]:
write_10X_h5(fernando_adata, '/mnt/sdb/scz_meta_analysis_processed/fernando_brennand/cellbender/fernando.h5')

In [78]:
fernando_adata

AnnData object with n_obs × n_vars = 147456000 × 33538
    obs: 'library'
    var: 'feature_types'

In [66]:
if not isinstance(adata.X, csr_matrix):
    adata.X = csr_matrix(adata.X)
